# Lecția 11 - Protocol Agent-la-Agent (A2A)


## Configurare


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Ce este Protocolul A2A?

**Protocolul Agent-la-Agent (A2A)** este un standard deschis care permite agenților AI să comunice,
să se descopere reciproc și să colaboreze — chiar și atunci când sunt construiți pe cadre diferite sau găzduiți
de servicii diferite.

Concepte cheie:

- **Descoperire** – Agenții publică o *Carte a Agentului* care descrie capacitățile lor, facilitând astfel
  găsirea specialistului potrivit pentru o sarcină de către alți agenți (sau organizatori).
- **Transmiterea mesajelor** – Agenții schimbă mesaje structurate printr-un protocol comun, astfel încât o
  cerere de la un agent să poată fi înțeleasă și îndeplinită de altul indiferent de implementarea internă.
- **Ciclul de viață al sarcinii** – A2A definește stări precum *trimisă*, *în curs*, *finalizată* și
  *eșuată*, oferind organizatorului vizibilitate completă asupra modului în care evoluează o sarcină delegată.

În această lecție simulăm colaborarea în stil A2A prin conectarea a trei agenți specializați în domeniul călătoriilor
într-un flux de lucru în care fiecare agent aduce expertiza sa și transmite rezultatele următorului.


## Crearea agenților de turism specializați


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Colaborare Multi-Agent prin Flux de Lucru

Conectăm cei trei agenți într-un flux de lucru secvențial care reflectă transmiterea mesajelor A2A:

1. **CurrencyExchangeAgent** primește cererea utilizatorului și produce ghidaj pentru valută.
2. **ActivityPlannerAgent** primește contextul îmbogățit și adaugă recomandări de activități.
3. **TravelManagerAgent** sintetizează ambele intrări într-un raport final de călătorie.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Înțelegerea A2A în producție

Într-un mediu de producție, protocolul A2A deblochează scenarii puternice între servicii:

| Capacitate | Descriere |
|---|---|
| **Interoperabilitate între framework-uri** | Un agent construit cu un framework poate delega sarcini către un agent construit cu orice alt framework compatibil A2A, permițând interoperabilitatea reală între organizații. |
| **Granițe de serviciu** | Agenții pot exista în microservicii separate, regiuni de cloud sau chiar în organizații diferite, colaborând totuși fără probleme. |
| **Descoperire dinamică** | Un orchestrator poate interoga un registru Agent Card la timpul execuției pentru a găsi specialistul cel mai potrivit pentru o anumită sub-sarcină. |
| **Streaming și notificări push** | A2A suportă Server-Sent Events (SSE) pentru actualizări în timp real ale progresului și notificări push pentru sarcini de durată lungă. |

Fluxul de lucru construit mai sus este o versiune simplificată, în proces, a acestui model. Într-o implementare reală, fiecare agent ar expune un endpoint HTTP, ar publica un Agent Card și ar comunica prin protocolul JSON-RPC A2A.


## Rezumat

În această lecție ai învățat:

1. **Ce este protocolul A2A** — un standard deschis pentru descoperirea, comunicarea și gestionarea sarcinilor între agenți.
2. **Cum să creezi agenți specializați** — un agent pentru schimb valutar, un agent planificator de activități și un coordonator Travel Manager.
3. **Cum să conectezi agenții într-un flux de lucru** — folosind `WorkflowBuilder` pentru a modela transmiterea secvențială a mesajelor între agenți.
4. **Cum funcționează A2A în producție** — facilitând colaborarea între diferite framework-uri și servicii, cu descoperire dinamică și actualizări în streaming.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
